# Phase 1 — Pure Python baseline and profiling

This notebook completes the assignment's **pure Python nested-loop element-wise cosine similarity** requirement. It loads the existing 20K corpus, constructs a **small, fixed TF-IDF vocabulary** for tractable Python-loop profiling, validates cosine scores against NumPy, runs `cProfile` and `line_profiler`, and saves raw measurements plus figures to Drive.

**Important:** The earlier 359.895 ms baseline used SciPy CSR operations inside a Python loop. This notebook is a distinct *pure Python arithmetic* baseline; compare only under the same vector representation, dimensionality, corpus subset, query and hardware.

No synthetic timing results are included. Run all cells to generate real figures.

In [ ]:
%pip -q install scikit-learn pandas matplotlib line_profiler

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, time, json, csv, math, statistics, cProfile, pstats, io, platform
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

ROOT=Path('/content/drive/MyDrive/HP-SS-Engine')
CORPUS=ROOT/'data/raw/qwen_corpus_20k_progress.jsonl'
OUT=ROOT/'benchmarks/raw_results'
PROF=ROOT/'benchmarks/profiling'
FIG=ROOT/'figures'
for d in (OUT,PROF,FIG): d.mkdir(parents=True,exist_ok=True)

rows=[json.loads(line) for line in CORPUS.open(encoding='utf-8') if line.strip()]
rows.sort(key=lambda r:r['id'])
assert len(rows)==20000 and [r['id'] for r in rows]==list(range(20000))
texts=[r['response'] for r in rows]
query_ids=list(range(0,20000,200))[:100]
queries=[rows[i]['prompt'] for i in query_ids]
print('Docs:',len(texts),'Queries:',len(queries))
print('Python:',sys.version.split()[0], 'CPU:',platform.processor(), 'CPU count:',os.cpu_count())
print('GPU not used: these calculations run on CPU')

In [ ]:
# Fixed dimensionality: Python loops over the full vocabulary are expensive.
# Keep the same 512 features for all corpus sizes so scale comparisons are fair.
N_FEATURES=512
vectorizer=TfidfVectorizer(max_features=N_FEATURES, lowercase=True, norm='l2',dtype=np.float32)
X=vectorizer.fit_transform(texts)
Q=vectorizer.transform(queries)
D=X.toarray()   # 20K x <=512 float32, ~41 MB
U=Q.toarray()
print('Dense TF-IDF shape:',D.shape,'dtype:',D.dtype,'memory MiB:',round(D.nbytes/2**20,2))
print('Vocabulary size:',len(vectorizer.vocabulary_))
# Convert only the rows required for Python-loop evaluation to lists.
# Conversion is setup, not included in per-query timing.
D_py=D.tolist()
U_py=U.tolist()

In [ ]:
def pure_python_cosine(a,b):
    # Deliberately naive: three explicit element-wise loops.
    dot=0.0
    for j in range(len(a)):
        dot += a[j]*b[j]
    aa=0.0
    for j in range(len(a)):
        aa += a[j]*a[j]
    bb=0.0
    for j in range(len(b)):
        bb += b[j]*b[j]
    if aa==0.0 or bb==0.0:
        return 0.0
    return dot/(math.sqrt(aa)*math.sqrt(bb))

def pure_python_topk(docs,query,k=10):
    scored=[]
    for i in range(len(docs)):
        score=pure_python_cosine(docs[i],query)
        scored.append((score,i))
    scored.sort(key=lambda x:(-x[0],x[1]))
    return scored[:k]

def numpy_reference_topk(docs,query,k=10):
    # Vectorizer L2-normalized TF-IDF vectors, so dot product = cosine.
    scores=docs@query
    order=np.lexsort((np.arange(len(scores)),-scores))[:k]
    return [(float(scores[i]),int(i)) for i in order]

# Verify individual scores and ranking with float tolerance.
for qi in range(3):
    for i in (0,1,10,99):
        a=D_py[i]; b=U_py[qi]
        got=pure_python_cosine(a,b)
        denom=np.linalg.norm(D[i])*np.linalg.norm(U[qi])
        expected=float(np.dot(D[i],U[qi])/denom) if denom else 0.0
        assert abs(got-expected)<1e-5,(i,qi,got,expected)
    got=pure_python_topk(D_py[:100],U_py[qi],10)
    ref=numpy_reference_topk(D[:100],U[qi],10)
    assert [i for _,i in got]==[i for _,i in ref],(qi,got,ref)
print('Correctness passed: scores within 1e-5 and top-10 IDs match for 3 queries / 100 docs')

In [ ]:
# Function-level profiling, 1000 docs / one query. Profiling overhead is excluded from timing benchmark.
prof=cProfile.Profile()
prof.enable()
pure_python_topk(D_py[:1000],U_py[0],10)
prof.disable()
s=io.StringIO()
pstats.Stats(prof,stream=s).strip_dirs().sort_stats('cumtime').print_stats(30)
profile_text=s.getvalue()
print(profile_text)
(PROF/'pure_python_1k_cprofile.txt').write_text(profile_text,encoding='utf-8')

# Line-level profiling of pure Python cosine (and its caller).
from line_profiler import LineProfiler
lp=LineProfiler()
lp.add_function(pure_python_cosine)
lp.add_function(pure_python_topk)
lp.enable_by_count()
pure_python_topk(D_py[:1000],U_py[0],10)
lp.disable_by_count()
s=io.StringIO()
lp.print_stats(stream=s)
line_text=s.getvalue()
print(line_text)
(PROF/'pure_python_1k_line_profile.txt').write_text(line_text,encoding='utf-8')
print('Saved both profiler outputs:',PROF)

In [ ]:
# Timed baseline. Start with 100/500/1000; no profiler active.
# Five queries and five repeats per scale; warmup is excluded.
SCALES=[100,500,1000]
QUERY_INDICES=list(range(5))
REPEATS=5
K=10
records=[]
for n in SCALES:
    docs=D_py[:n]
    pure_python_topk(docs,U_py[0],K) # warmup
    for qi in QUERY_INDICES:
        for repeat in range(REPEATS):
            t0=time.perf_counter_ns()
            got=pure_python_topk(docs,U_py[qi],K)
            elapsed=(time.perf_counter_ns()-t0)/1e6
            records.append(dict(implementation='pure_python_nested_loops',n_docs=n,
                n_features=D.shape[1],query_index=qi,repeat=repeat,top_k=K,
                elapsed_ms=elapsed,representation='tfidf_dense_float32_512features'))
    measured=[r['elapsed_ms'] for r in records if r['n_docs']==n]
    print(f'{n} docs: median {statistics.median(measured):.3f} ms; mean {statistics.mean(measured):.3f} ms')

csv_path=OUT/'pure_python_cosine_phase1_raw.csv'
with csv_path.open('w',newline='',encoding='utf-8') as f:
    w=csv.DictWriter(f,fieldnames=list(records[0]))
    w.writeheader();w.writerows(records)
print('Saved',len(records),'measurements to',csv_path)

In [ ]:
# Figures are generated from measured timings only.
summary=[]
for n in SCALES:
    a=[r['elapsed_ms'] for r in records if r['n_docs']==n]
    summary.append((n,statistics.median(a),statistics.mean(a),min(a),max(a)))
fig,ax=plt.subplots(figsize=(8,5))
ax.plot([x[0] for x in summary],[x[1] for x in summary],marker='o',linewidth=2)
ax.set_xlabel('Corpus size (documents)')
ax.set_ylabel('Median search latency (ms)')
ax.set_title('Pure Python cosine top-10: baseline scaling (512 TF-IDF features)')
ax.grid(alpha=.25)
fig.tight_layout()
fig_path=FIG/'phase1_pure_python_scaling.png'
fig.savefig(fig_path,dpi=180)
plt.show()
plt.close(fig)

fig,ax=plt.subplots(figsize=(8,5))
data=[[r['elapsed_ms'] for r in records if r['n_docs']==n] for n in SCALES]
ax.boxplot(data,tick_labels=[str(n) for n in SCALES])
ax.set_xlabel('Corpus size (documents)')
ax.set_ylabel('Search latency (ms)')
ax.set_title('Pure Python cosine top-10: 25 runs per corpus size')
ax.grid(axis='y',alpha=.25)
fig.tight_layout()
box_path=FIG/'phase1_pure_python_latency_boxplot.png'
fig.savefig(box_path,dpi=180)
plt.show()
plt.close(fig)
print('Saved figures:',fig_path,box_path)

## What to send back

Send the outputs of the **correctness**, **cProfile**, **line_profiler**, and **median timings** cells. The raw CSV, profiling text, and two PNG figures will already be saved under the project folder in Drive. We will then identify the top three bottlenecks, update the report and benchmark tables, and choose the appropriate scale for further baseline runs.

**Methodology note:** The earlier CSR/SciPy baseline and this 512-feature dense Python baseline use different representations; do not directly claim a speedup between them. Phase 2 must benchmark optimized implementations on the *same* 512-feature dense representation for a valid speedup comparison.